BAUDE-180


In [0]:
%sql
alter table `p-cdh`.primaryactive.agreement add column HighBalanceFlag boolean after LimitUtilizedAmount;
alter table `p-cdh`.primaryarchive.agreement add column HighBalanceFlag boolean after LimitUtilizedAmount;

DaysPastDue_Fill

In [0]:
%sql
UPDATE `p-cdh`.primaryactive.exposure
SET DaysPastDue = DaysPastDueWithMateriality
WHERE SourceSystem IN ('qca', 'arotron');

UPDATE `p-cdh`.primaryarchive.exposure
SET DaysPastDue = DaysPastDueWithMateriality
WHERE SourceSystem IN ('qca', 'arotron');

In [0]:
%sql
UPDATE `p-cdh`.derivativeactive.dexposure
SET DaysPastDue = DaysPastDueWithMateriality
WHERE SourceSystem IN ('qca', 'arotron');

UPDATE `p-cdh`.derivativearchive.dexposure
SET DaysPastDue = DaysPastDueWithMateriality
WHERE SourceSystem IN ('qca', 'arotron');

Backfill run for issecuredFlag in Dexposre

In [0]:
# text input, with a default value
dbutils.widgets.text("env", "p", "Environment")

In [0]:
# =============================================================================
# Backfill IsSecuredFlag for HAPS portfolios in derivativeactive.dexposure
# - Active table only  - updates ONLY IsSecuredFlag  - key: AgreementKey + BusinessDate
# =============================================================================
from pyspark.sql.functions import col, lit, try_divide

# ---- CONFIG -----------------------------------------------------------------
env=dbutils.widgets.get("env")
TARGET_TABLE = f"`{env}-cdh`.derivativeactive.dexposure"

HAPS_PORTFOLIOS = (
    'Orion X Securitisation DAC', 'Galaxy II Funding DAC', 'Cosmos Retail',
    'GAIA II PORTFOLIO WHOLESALE', 'DOMUS PORTFOLIO - RETAIL',
    'DOMUS PORTFOLIO - WHOLESALE', 'RHODIUM PORTFOLIO - RETAIL',
    'RHODIUM PORTFOLIO - WHOLESALE', 'GAIA I PORTFOLIO', 'GAIA II PORTFOLIO RETAIL'
)
haps_sql_list = ", ".join(f"'{p}'" for p in HAPS_PORTFOLIOS)

# ---- HAPS SECURED-FLAG CALCULATION (per BusinessDate) -----------------------
def compute_haps_flag(run_date):
    """Returns DataFrame[AgreementKey, is_secured_flag] for one BusinessDate."""
    # Agreements with NO active thesis collateral on this date
    dcollateral_notactive = spark.sql(f"""
        SELECT DISTINCT c.AgreementKey
        FROM `{env}-cdh`.primary.collateral c
        WHERE c.SourceSystem = 'thesis'
          AND c.BusinessDate = '{run_date}'
          AND c.PortfolioName IN ({haps_sql_list})
          AND NOT EXISTS (
              SELECT 1 FROM `{env}-cdh`.primary.collateral dc
              WHERE dc.SourceSystem = 'thesis'
                AND dc.RealEstateCollateralStatusCode = 'Ενεργή'
                AND dc.BusinessDate = c.BusinessDate
                AND dc.AgreementKey  = c.AgreementKey)
    """)

    # Agreements whose total LVNetBase = 0
    dmetrics_zeroLV = spark.sql(f"""
        SELECT AgreementKey
        FROM `{env}-cdh`.derivative.danalyticsmetrics
        WHERE BusinessDate = '{run_date}' AND PortfolioName IN ({haps_sql_list})
        GROUP BY AgreementKey HAVING SUM(COALESCE(LVNetBase, 0)) = 0
    """)

    # TLC per agreement from the exposure table itself
    exposure_TLC = spark.sql(f"""
        SELECT AgreementKey,
               SUM(try_cast(OutstandingTotalBalance AS DECIMAL(20,2))) AS TLC
        FROM {TARGET_TABLE}
        WHERE BusinessDate = '{run_date}' AND PortfolioName IN ({haps_sql_list})
        GROUP BY AgreementKey
    """)

    dmetrics_CREVBaseTLC = spark.sql(f"""
        SELECT AgreementKey, SUM(COALESCE(CREVBaseTLC, 0)) AS TotalCREVBaseTLC
        FROM `{env}-cdh`.derivative.danalyticsmetrics
        WHERE BusinessDate = '{run_date}' AND PortfolioName IN ({haps_sql_list})
        GROUP BY AgreementKey HAVING SUM(COALESCE(CREVBaseTLC, 0)) < 10000
    """)

    # CREV < 10K AND coverage < 10%
    dmetrics_CrevCoverage = (
        exposure_TLC.join(dmetrics_CREVBaseTLC, 'AgreementKey', 'left')
        .withColumn('CrevCoverage',
                    try_divide(dmetrics_CREVBaseTLC.TotalCREVBaseTLC, exposure_TLC.TLC))
        .filter(col('CrevCoverage') < 0.10)
        .select(exposure_TLC.AgreementKey)
    )

    union_df = dcollateral_notactive.union(dmetrics_zeroLV).union(dmetrics_CrevCoverage)
    Haps_UnsecuredAgreements = (
        union_df.select('AgreementKey').distinct().withColumn('is_secured_flag', lit(False))
    )

    # Agreements WITH active thesis collateral
    dcollateral_active = spark.sql(f"""
        SELECT DISTINCT ex.AgreementKey
        FROM {TARGET_TABLE} ex
        INNER JOIN `{env}-cdh`.primary.collateral c
            ON c.AgreementKey = ex.AgreementKey AND c.BusinessDate = ex.BusinessDate
        WHERE c.RealEstateCollateralStatusCode = 'Ενεργή'
          AND c.SourceSystem = 'thesis'
          AND ex.BusinessDate = '{run_date}' AND ex.PortfolioName IN ({haps_sql_list})
    """)

    # active-collateral agreements NOT already marked unsecured -> secured (True)
    secured_agreements = dcollateral_active.join(
        Haps_UnsecuredAgreements, 'AgreementKey', 'left_anti'
    ).withColumn('is_secured_flag', lit(True))

    return Haps_UnsecuredAgreements.union(secured_agreements).distinct()

# ---- ROLLBACK POINT ---------------------------------------------------------
ver = spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1").first()["version"]
print(f"RESTORE point (version before backfill): {ver}")
print(f"-> rollback with: RESTORE TABLE {TARGET_TABLE} TO VERSION AS OF {ver}")

# ---- DATES TO BACKFILL (every HAPS date in the active table) ----------------
dates = [r.BusinessDate for r in spark.sql(f"""
    SELECT DISTINCT BusinessDate
    FROM {TARGET_TABLE}
    WHERE PortfolioName IN ({haps_sql_list})
    ORDER BY BusinessDate
""").collect()]
print(f"{len(dates)} HAPS business dates to backfill")

# ---- LOOP + MERGE (spark.sql, only the IsSecuredFlag column) ----------------
for d in dates:
    compute_haps_flag(d).createOrReplaceTempView("haps_flags")
    spark.sql(f"""
        MERGE INTO {TARGET_TABLE} AS t
        USING haps_flags AS s
          ON t.AgreementKey = s.AgreementKey
         AND t.BusinessDate = DATE'{d}'
        WHEN MATCHED THEN UPDATE SET t.IsSecuredFlag = s.is_secured_flag
    """)
    print(f"  {d}: merged {spark.table('haps_flags').count()} agreements")

print("DONE")


Newsphone Auction Report Content Backfill

In [0]:
import os, sys, boto3, time
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("newsphone_arc_backfill").getOrCreate()
spark.conf.set("spark.sql.ansi.enabled", "false")

# ============================================================
# ONLY THING TO CHANGE BETWEEN ENVIRONMENTS:
ENV_PREFIX = "p"            # "d" = dev, "p" = prod, "qa" = qa
# ============================================================
CATALOG = f"`{ENV_PREFIX}-cdh`"

# ---- AWS credentials from Databricks secret scope ----
SECRETS_SCOPE = "aws-databricks"
_keys = dbutils.secrets.list(scope=SECRETS_SCOPE)
_aws_key = [k.key for k in _keys][0]
_aws_secret = dbutils.secrets.get(scope=SECRETS_SCOPE, key=_aws_key)
os.environ["AWS_ACCESS_KEY_ID"] = _aws_key
os.environ["AWS_SECRET_ACCESS_KEY"] = _aws_secret
os.environ["AWS_DEFAULT_REGION"] = "eu-central-1"

# ---- Auto-discover the helpers folder (works for ANY user / bundle / repo) ----
def find_helpers_base():
    """
    Locate .../src/helpers by finding the folder that contains
    src/helpers/lng_lnd/landing_helpers.py, searching common roots.
    Works regardless of which user deployed the bundle or cloned the repo.
    """
    search_roots = [
        "/Workspace/Repos",     # Databricks Repos (any user)
        "/Workspace/Users",     # bundles / user workspaces
        "/Workspace/Shared",    # shared location
        "/Workspace",           # fallback (slowest)
    ]
    seen = set()
    for base in search_roots:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if root in seen:
                continue
            seen.add(root)
            if root.replace("\\", "/").endswith("src/helpers/lng_lnd") and "landing_helpers.py" in files:
                return os.path.dirname(root)   # -> .../src/helpers
    raise RuntimeError(
        "Could not locate 'src/helpers/lng_lnd/landing_helpers.py' under /Workspace. "
        "Set HELPERS_BASE manually to the absolute path of your repo's src/helpers folder."
    )

# If you already know the path, you can skip discovery by setting it directly:
# HELPERS_BASE = "/Workspace/.../src/helpers"
HELPERS_BASE = find_helpers_base()
print("helpers found at:", HELPERS_BASE)

for sub in ["lng_lnd", "authentication_helpers", "curated_helpers"]:
    p = f"{HELPERS_BASE}/{sub}"
    if p not in sys.path:
        sys.path.append(p)

from landing_helpers import *          # noqa
from landed_helpers import *           # noqa
from newsphone import *               # noqa
from external_file_helpers import *    # noqa

# ---- Derived config (do not edit) ----
SCHEMA_LNG   = "`landing`"
SCHEMA_LND   = "`landed`"
TABLE_NAME   = "newsphone_auction_report_content"
FOLDER_NAME  = "newsphone_auction_report_content"
SOURCE_BUCKET    = f"{ENV_PREFIX}-ec1-cepal-landing-zone"
INCOMING_PREFIX  = "incoming/newsphone/"
PROCESSED_PREFIX = "incoming/newsphone/processed/"
FILE_AREA    = "AuctionReportContent"

TABLE_SCHEMA   = '{"UniqueCode": "NVARCHAR(200)", "FileContent": "NVARCHAR(MAX)"}'
PRIMARY_KEYS   = ["UniqueCode"]
INGESTION_CASE = "incremental"

cdh_location_lng = f"{CATALOG}.{SCHEMA_LNG}.{FOLDER_NAME}"
cdh_location_lnd = f"{CATALOG}.{SCHEMA_LND}.{TABLE_NAME}"

s3 = boto3.client("s3", region_name="eu-central-1")

print("setup done | env:", ENV_PREFIX)
print("ansi:", spark.conf.get("spark.sql.ansi.enabled"))
print("landing:", cdh_location_lng)
print("landed :", cdh_location_lnd)
print("bucket :", SOURCE_BUCKET)


In [0]:
def list_incoming_arc_files():
    """Top-level AuctionReportContent_*.csv files in incoming/newsphone/ (excludes processed/)."""
    pg = s3.get_paginator("list_objects_v2")
    keys = []
    for page in pg.paginate(Bucket=SOURCE_BUCKET, Prefix=INCOMING_PREFIX, Delimiter="/"):
        for o in page.get("Contents", []):
            f = o["Key"].split("/")[-1]
            if f.startswith(FILE_AREA) and f.endswith(".csv"):
                keys.append(f)
    return keys

def date_key(f):
    return f.split("_")[-1].split(".csv")[0]          # YYYYMMDD

def business_date_from_file(f):
    d = date_key(f)
    return f"{d[0:4]}-{d[4:6]}-{d[6:8]}"              # YYYY-MM-DD

def archive_and_remove(f):
    s3.copy_object(
        Bucket=SOURCE_BUCKET,
        CopySource={"Bucket": SOURCE_BUCKET, "Key": f"{INCOMING_PREFIX}{f}"},
        Key=f"{PROCESSED_PREFIX}{f}",
    )
    s3.delete_object(Bucket=SOURCE_BUCKET, Key=f"{INCOMING_PREFIX}{f}")

print("functions defined")


In [0]:
# Pre-create the landing table with COMPATIBLE Delta retention so the loader's
# append path is used instead of its create branch (which sets a 35-day
# deletedFileRetentionDuration that fails validation against the 30-day default).
# IF NOT EXISTS => safe to run anywhere; does nothing if the table already exists.

if spark.catalog.tableExists(cdh_location_lng):
    print(f"Landing table already exists: {cdh_location_lng} (no action)")
else:
    spark.sql(f"""
        CREATE TABLE {cdh_location_lng} (
            UniqueCode STRING,
            FileContent STRING,
            CorruptRecordFlag BOOLEAN,
            file_date DATE,
            file_name STRING,
            Cluster_0 STRING
        )
        USING delta
        CLUSTER BY (Cluster_0)
        TBLPROPERTIES (
            'delta.deletedFileRetentionDuration' = 'interval 5 weeks',
            'delta.logRetentionDuration' = 'interval 5 weeks',
            'delta.columnMapping.mode' = 'name'
        )
    """)
    print(f"Landing table created: {cdh_location_lng}")

spark.sql(f"DESCRIBE {cdh_location_lng}").show(50, truncate=False)


In [0]:
files = sorted(list_incoming_arc_files(), key=date_key)
total = len(files)
print(f"Backfilling {total} files oldest -> newest")

for i, fname in enumerate(files, start=1):
    bdate = business_date_from_file(fname)
    print(f"[{i}/{total}] {fname}  {bdate}")

    extfile_args = {
        "table_schema": TABLE_SCHEMA,
        "file_name": fname,
        "schema_catalog": SCHEMA_LNG,
        "source_bucket": SOURCE_BUCKET,
        "folder_name": FOLDER_NAME,
        "cdh_location_lng": cdh_location_lng,
    }
    landing_df = load_newsphone_external_file(spark, extfile_args)
    load_table_to_landed(
        spark, landing_df, TABLE_NAME, cdh_location_lnd,
        INGESTION_CASE, PRIMARY_KEYS, bdate,
        schema_name="newsphone", group_name="newsphone",
        ingestion_date=bdate, env_prefix=ENV_PREFIX,
    )
    archive_and_remove(fname)

    spark.catalog.clearCache()     

print("BACKFILL COMPLETE")
